# Serve Qwen3.8-27B on Kaggle free tier

Gives you an **OpenAI-compatible endpoint** on a public HTTPS URL that your laptop can call.

## How to use it

**Once, in the sidebar:** set **Accelerator = GPU T4 x2** and **Internet = ON**.

**Then just click `Run All`.** Nothing else. The URL and key appear in cell 4.

**To stop:** click **Stop Session** in the sidebar. Do this the moment you finish, because the free quota burns while the session runs, whether or not you are using it.

It also stops itself after `RUN_MINUTES` so a forgotten tab cannot drain your quota overnight.

## Quota

About **30 GPU-hours a week**, sessions capped near 12 hours. The first run spends 10 to 20 minutes downloading 19.5 GB. **Cell 6 shows how to avoid paying that again on every restart.**

In [ ]:
# ============================================================
#  CONFIG - the only cell you might want to edit
# ============================================================

RUN_MINUTES = 120     # auto-shutdown after this long. Protects your quota.

MODEL       = "philbert440/Qwen3.8-27B-W4A16-AWQ"   # ~19.5 GB, fits 2x T4
# Fallback if the above fails to load:
# MODEL     = "cyankiwi/Qwen3.8-27B-AWQ-INT4"       # ~21 GB, more widely used

MAX_LEN     = 8192    # lower to 4096 if you hit CUDA out-of-memory

# If you have saved the model as a Kaggle Dataset (see cell 6), put its path
# here to skip the download entirely. Leave as None to download from the hub.
LOCAL_MODEL_DIR = None

print(f"Will serve {MODEL}")
print(f"Auto-shutdown after {RUN_MINUTES} minutes.")

In [ ]:
# 1. Check the hardware, then install. Fails fast if the sidebar is wrong.

import torch, subprocess, sys

n = torch.cuda.device_count()
print(f"GPUs visible: {n}")
assert n >= 2, "Set Accelerator to 'GPU T4 x2' in the sidebar, then Run All again."
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

print("\nInstalling vllm and cloudflared (a few minutes)...")
!pip install -q vllm 2>&1 | tail -2
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("ready")

In [ ]:
# 2. Start the model server in the background.
#
#    T4 is Turing: no bfloat16, no FlashAttention-2, no Marlin kernels.
#    Hence float16, plain awq, and enforce-eager.

import subprocess, secrets, time

PORT    = 8000
API_KEY = secrets.token_urlsafe(24)
model_path = LOCAL_MODEL_DIR or MODEL

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--served-model-name", "qwen",
    "--api-key", API_KEY,
    "--port", str(PORT),
    "--tensor-parallel-size", "2",
    "--dtype", "float16",
    "--quantization", "awq",
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.92",
    "--enforce-eager",
    "--limit-mm-per-prompt", "image=0,video=0",
]

LOG = "/kaggle/working/vllm.log"
server = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("starting vLLM, pid", server.pid)
if not LOCAL_MODEL_DIR:
    print("first run downloads ~19.5 GB, so 10-20 minutes is normal")

In [ ]:
# 3. Wait for it, then open the tunnel and print what the laptop needs.

import requests, re, time

def die(msg):
    print(msg + "\n--- last 40 log lines ---")
    print("".join(open(LOG).readlines()[-40:]))
    raise SystemExit(msg)

deadline, shown = time.time() + 45*60, 0
while time.time() < deadline:
    if server.poll() is not None:
        die("vLLM exited during startup.")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models",
                        headers={"Authorization": f"Bearer {API_KEY}"}, timeout=5).ok:
            print("\nmodel loaded")
            break
    except requests.RequestException:
        pass
    lines = open(LOG).readlines()
    for ln in lines[shown:]:
        if any(k in ln for k in ("Loading", "error", "Error", "CUDA", "out of memory", "%|")):
            print(ln.rstrip()[:150])
    shown = len(lines)
    time.sleep(20)
else:
    die("timed out waiting for vLLM.")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(40):
    time.sleep(3)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",
                  open("/kaggle/working/tunnel.log").read())
    if m:
        PUBLIC = m.group(0); break
assert PUBLIC, "tunnel failed; see /kaggle/working/tunnel.log"

r = requests.post(PUBLIC + "/v1/chat/completions",
                  headers={"Authorization": f"Bearer {API_KEY}"},
                  json={"model": "qwen", "max_tokens": 8, "temperature": 0,
                        "messages": [{"role": "user", "content": "say: ready"}]},
                  timeout=180)

print("\n" + "="*66)
print("  ENDPOINT LIVE" if r.ok else f"  TUNNEL UP BUT TEST FAILED ({r.status_code})")
print("="*66)
print(f"  OPENAI_BASE_URL = {PUBLIC}/v1")
print(f"  OPENAI_API_KEY  = {API_KEY}")
print(f"  model name      = qwen")
print("="*66)
print("  Both change every session. Keep them in a .env, never in git.")

In [ ]:
# 4. Hold the endpoint open, then shut down automatically.
#
#    Interrupt this cell at any time to release the GPU early.
#    Then click Stop Session in the sidebar to end the billing period.

import time, datetime

end = time.time() + RUN_MINUTES * 60
print(f"Serving until {datetime.datetime.fromtimestamp(end):%H:%M:%S} "
      f"({RUN_MINUTES} min). Interrupt this cell to stop sooner.\n")

try:
    while time.time() < end:
        if server.poll() is not None:
            print("vLLM died:")
            print("".join(open(LOG).readlines()[-30:]))
            break
        left = int((end - time.time()) / 60)
        print(f"{datetime.datetime.now():%H:%M:%S}  alive, {left} min left  |  {PUBLIC}")
        time.sleep(300)
finally:
    for p in (tunnel, server):
        try: p.terminate()
        except Exception: pass
    time.sleep(5)
    print("\n" + "="*66)
    print("  Model stopped. GPU MEMORY is free.")
    print("")
    print("  YOUR QUOTA IS STILL RUNNING.")
    print("  Kaggle charges for an ATTACHED GPU session, not for GPU use.")
    print("  Click STOP SESSION in the sidebar. That is the only thing")
    print("  that stops the meter. The last cell is a fallback.")
    print("="*66)

## Cell 6: skip the 19.5 GB download on every restart

Re-downloading costs 10 to 20 minutes of GPU quota **every single session**. Pay it once instead.

**Once:** run the cell below, then in the sidebar use **Output → Create Dataset** and save `/kaggle/working/model`. Give it a name.

**Every session after:** attach that dataset via **Add Input**, and set in the config cell:

```python
LOCAL_MODEL_DIR = "/kaggle/input/<your-dataset-name>/model"
```

Startup then drops to roughly two minutes.

```python
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL, local_dir="/kaggle/working/model",
                  local_dir_use_symlinks=False)
print("downloaded. Now: Output -> Create Dataset")
```

---

## Using it from the laptop

```python
from openai import OpenAI
client = OpenAI(base_url="<OPENAI_BASE_URL from cell 3>",
                api_key="<OPENAI_API_KEY from cell 3>")
print(client.chat.completions.create(
    model="qwen",
    messages=[{"role": "user", "content": "hello"}],
).choices[0].message.content)
```

## If it fails

| Symptom | Fix |
|---|---|
| CUDA out of memory | Set `MAX_LEN = 4096`. Still failing: switch `MODEL` to the cyankiwi fallback, or drop to a smaller model on one T4 |
| Only 1 GPU visible | Accelerator is not **T4 x2** |
| Download stuck at 0% | **Internet** is off in the sidebar |
| Marlin / FlashAttention error | Expected on Turing; `--quantization awq` and `--enforce-eager` already avoid both |
| URL stopped working | Session ended. Run All again; URL and key are new each time |

## Honest caveats

- **This has not been run on real hardware yet.** The settings follow from documented T4 limits and the measured 19.5 GB weight size. The first run is the test.
- **Roughly 5.5 GB per GPU is left for KV cache.** That is tight. The fallback is a smaller model on a single T4, which is comfortable and sufficient, since the LLM here only handles language and never computes a number or decides an action.
- **Quota is the real constraint**, which is why the evaluation caches and replays model responses instead of depending on a live endpoint.

## What actually stops the quota

Easy to get wrong, so stated plainly.

| Action | Frees GPU memory | **Stops quota** |
|---|---|---|
| Timer expires, or you interrupt the cell | Yes | **No** |
| Closing the browser tab | No | **No** |
| **Stop Session** in the sidebar | Yes | **Yes** |
| Kaggle's own idle timeout | Yes | Yes, eventually |

Kaggle bills for a GPU session being **attached**, not for the GPU doing work. The auto-shutdown above protects you from a runaway model. It does **not** end the session.

Kaggle does stop idle interactive sessions by itself, so a forgotten tab will not run indefinitely. But that only starts once the notebook stops executing, and the exact delay is not something to rely on.

**Rule: when you finish, click Stop Session. Everything else is a safety net, not a substitute.**


In [ ]:
# OPTIONAL FALLBACK: end the container from inside the notebook.
#
# Kills every process in the container, which terminates the session and stops
# the quota. Blunt: anything unsaved in /kaggle/working is lost and the notebook
# will look like it crashed. That is expected.
#
# The Stop Session button is better. Use this only if you cannot reach it.

CONFIRM = False   # set to True, then run this cell

if CONFIRM:
    import os, signal
    print("terminating container...")
    os.kill(1, signal.SIGKILL)
else:
    print("Not armed. Set CONFIRM = True to actually terminate.")
    print("Prefer Stop Session in the sidebar.")
